# Preprocessing

## Importing modules

In [ ]:
import pandas as pd
import numpy as np


## 1. Importing and merging data

- Importing the clinical dataset as X_clinical, labels data as Y.
- We are merging them as clinical_dataset: lef join on 'ID'. The EDA reveals that IDs in both datasets are unique and all patients in the X_clinical dataset have a record on the labels dataset and vice versa.
- We are setting the index to 'ID'.

In [ ]:
X_clinical = pd.read_csv('../data/raw/X_train/clinical_train.csv')
Y = pd.read_csv('../data/raw/target_train.csv')

clinical_dataset = pd.merge(X_clinical, Y, how='left', on='ID', validate='one_to_one')
clinical_dataset.set_index('ID', inplace=True)
print("Size of the merged dataset: {clinical_dataset.shape}".format(clinical_dataset=clinical_dataset))
clinical_dataset.head()

## 2. Mising data

### 2.1 Missing data in the labesl

In [ ]:
Y_missing = pd.DataFrame({'Number of missing data in the labels' : clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum(), 'Proportion of missing data in the labels (%)' : (clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum()/clinical_dataset[['OS_YEARS', 'OS_STATUS']].shape[0])*100})
Y_missing

As seen in the EDA 150 values are missing in the labels' dataset, the analysis of these values shows that patients with missing values in OS_YEARS are the same missing values in OS_STATUS. These patients do not brigs any information to our study, we are dropping them.

In [ ]:
clinical_dataset.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
print("New size of the clinical dataset: {clinical_dataset.shape}".format(clinical_dataset=clinical_dataset))
clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum()

### 2.2 Missing data in the features

A quick reminder of what we found during the EDA :

There is a lot of missing values for the MONOCYTES (18.1%) CYTOGENETICS (11.6%) and WBC (8.2%) features.
- The REL and UOXF centers did not collect any MONOCYTES data. We won't be able to replace these missing values with values estimated per center.
- Missing data are over-represented in some centers: HMS & DUTH.
- Missing data in the MONOCYTES, WBC, ANC and PLT features seem to vary by CENTER
- Missing data in the BM_BLAST and HB features seem to be more randomly distributed \
\
**In the clinical dataset, missing values are not randomly distributed. We must take this into account when replacing missing data.**

We will estimate the missing values with the median in the pipline to avoid any leaks.

## 3. CYTOGENETICS

We are creating a feature to indicate whether the karyotype is normal or abnormal. \
The feature takes a dataframe with the karyotype and return the dataframe with four new features:
- CYTO_NORMAL : True if the karyotype is normal, False otherwise.
- CYTO_ABNORMAL: True if the karyotype is not normal (including if it is complex) and not an na, False otherwise.
- CYTO_COMPLEX: True if the karyotype specify complex, False otherwise.
- CYTO_MISSING: True if the karyotype is missing (na), False otherwise.

In [ ]:
def add_cytogenetics_flags(data_frame):
    cyto = data_frame['CYTOGENETICS']
    # Creating masks
    is_missing = cyto.isna()
    is_normal = cyto.str.match(r'46,(xx|xy)(\[\d+\])?$', case=False, na=False)
    is_complex = cyto.str.match('complex', case=False, na=False)

    # Creating the normal feature
    data_frame['CYTO_NORMAL'] = is_normal & ~is_missing
    # Creating the complex feature
    data_frame['CYTO_COMPLEX'] = is_complex & ~is_missing
    # Creating the abnormal feature
    data_frame['CYTO_ABNORMAL'] = ~is_normal & ~is_missing
    # Creating de is missing feature
    data_frame['CYTO_MISSING'] = is_missing

    return data_frame

In [ ]:
df_test =add_cytogenetics_flags(clinical_dataset)
df_test[['CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING']].value_counts()

## 4. Log transformation

Adding missing value indicator.

In [ ]:
hemo_param = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
for var in hemo_param:
    clinical_dataset[var+'_MISSING'] = clinical_dataset[var].isna()

clinical_dataset.head()

Transforming BM_BLAST, WBC, ANC, MONOCYTES and PLT with log1p.

In [ ]:
hemo_param_log = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'PLT']
for var in hemo_param_log:
    clinical_dataset[var] = np.log1p(clinical_dataset[var])
    clinical_dataset.rename(columns={var : var+'_LOG'}, inplace=True)

clinical_dataset_log.head()